# BuzzASR Filipino on a Colab TPU through a Rust gateway

Select **Runtime → Change runtime type → TPU** before running. This notebook performs a local proof only: it does not create a public tunnel or production endpoint.

In [ ]:
# Install a matched PyTorch/XLA pair. The first run intentionally restarts the runtime.
import os, pathlib, subprocess, sys
marker = pathlib.Path('/content/.buzzasr_xla_installed')
if not marker.exists():
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'torch==2.8.0', 'torch_xla[tpu]==2.8.0',
                    'transformers==4.57.1', 'safetensors>=0.6,<0.7', 'numpy>=2,<3',
                    '-f', 'https://storage.googleapis.com/libtpu-releases/index.html'], check=True)
    marker.touch()
    print('Installed. The runtime will restart; rerun this cell afterward.')
    os.kill(os.getpid(), 9)
else:
    print('Pinned TPU packages are installed.')

In [ ]:
# Prove that this session has a real XLA device before downloading the model.
import torch, torch_xla
device = torch_xla.device()
assert str(device).startswith('xla'), f'TPU not available: {device}'
probe = (torch.ones((2, 2), device=device) @ torch.ones((2, 2), device=device)).cpu()
assert probe.tolist() == [[2.0, 2.0], [2.0, 2.0]]
print({'status': 'TPU_READY', 'device': str(device), 'torch': torch.__version__})

In [ ]:
# Upload the historical Colab source archive from backend/model/experiments/colab.
from google.colab import files
import pathlib, shutil, zipfile
uploaded = files.upload()
archive_name = next((name for name in uploaded if name.endswith('.zip')), None)
assert archive_name, 'Upload buzzasr-colab-sources.zip'
workspace = pathlib.Path('/content/agora-buzzasr')
if workspace.exists():
    shutil.rmtree(workspace)
workspace.mkdir()
with zipfile.ZipFile(archive_name) as archive:
    archive.extractall(workspace)
assert (workspace / 'colab/buzzasr_tpu_worker.py').exists()
assert (workspace / 'backend/model/api/Cargo.toml').exists()
print({'status': 'SOURCES_READY', 'workspace': str(workspace)})

In [ ]:
# Install Rust if necessary, then compile the gateway.
import os, pathlib, shutil, subprocess
if shutil.which('cargo') is None:
    subprocess.run('curl --proto =https --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal', shell=True, check=True)
os.environ['PATH'] = f"{pathlib.Path.home() / '.cargo/bin'}:{os.environ['PATH']}"
subprocess.run(['cargo', 'build', '--release', '--manifest-path',
                str(workspace / 'backend/model/api/Cargo.toml')], check=True)
gateway_binary = workspace / '.artifacts/rust-target/release/buzzasr-gateway'
assert gateway_binary.exists()
print({'status': 'RUST_READY', 'binary': str(gateway_binary)})

In [ ]:
# Start Rust. It starts Python as a persistent JSONL TPU worker and binds only after model readiness.
import os, pathlib, subprocess, time, requests
log_path = pathlib.Path('/content/buzzasr-gateway.log')
log_file = log_path.open('w')
env = os.environ.copy()
env.update({
    'PJRT_DEVICE': 'TPU',
    'BUZZASR_MODEL_ID': 'BuzzASR/filipino',
    'BUZZASR_WORKER_SCRIPT': str(workspace / 'colab/buzzasr_tpu_worker.py'),
    'BUZZASR_BIND': '127.0.0.1:4318',
})
gateway = subprocess.Popen([str(gateway_binary)], cwd=workspace, env=env, stdout=log_file, stderr=subprocess.STDOUT)
health = None
for _ in range(150):
    if gateway.poll() is not None:
        break
    try:
        response = requests.get('http://127.0.0.1:4318/health', timeout=2)
        if response.ok:
            health = response.json()
            break
    except requests.RequestException:
        pass
    time.sleep(2)
if health is None:
    log_file.flush()
    tail = log_path.read_text(errors='replace').splitlines()[-10:]
    raise RuntimeError('Gateway/model failed to become ready:\n' + '\n'.join(tail))
assert health['device'].startswith('xla')
print({'status': 'MODEL_READY', **health})

In [ ]:
# Full-path silence smoke test: Rust HTTP → JSONL → Python → TPU → Rust HTTP.
import base64, io, wave, requests
wav_bytes = io.BytesIO()
with wave.open(wav_bytes, 'wb') as wav:
    wav.setnchannels(1); wav.setsampwidth(2); wav.setframerate(16000)
    wav.writeframes(b'\x00\x00' * 16000)
response = requests.post('http://127.0.0.1:4318/transcribe', json={
    'audio_wav_base64': base64.b64encode(wav_bytes.getvalue()).decode('ascii')
}, timeout=310)
result = response.json()
assert response.ok and result['ok'] and result['device'].startswith('xla'), result
print({'status': 'TPU_MODEL_SMOKE_PASSED', **result})

In [ ]:
# Upload real audio; ffmpeg normalizes it to the gateway contract before transcription.
from google.colab import files
import base64, pathlib, subprocess, requests
audio_upload = files.upload()
source_audio = pathlib.Path(next(iter(audio_upload)))
normalized_audio = pathlib.Path('/content/buzzasr-input.wav')
subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(source_audio),
                '-ac', '1', '-ar', '16000', '-c:a', 'pcm_s16le', str(normalized_audio)], check=True)
payload = base64.b64encode(normalized_audio.read_bytes()).decode('ascii')
response = requests.post('http://127.0.0.1:4318/transcribe',
                         json={'audio_wav_base64': payload}, timeout=310)
result = response.json()
assert response.ok and result['ok'], result
print({'status': 'REAL_AUDIO_PASSED', **result})

In [ ]:
# Stop local processes when testing is complete.
gateway.terminate()
gateway.wait(timeout=15)
log_file.close()
print({'status': 'STOPPED'})